# Hands-on: Run the AIFS

This notebook describes how to run inference of the AIFS from an existing checkpoint.

This version runs a small **AIFS-ENS** model (O48 grid, ~10,000 grid points, ~9 million weights) on a **CPU**. The same notebook for a larger AIFS-ENS model (O96 grid, ~40,000 grid points, ~250 million weights) on a GPU is `inference-aifs-ens-o96-gpu.ipynb`. Both models forecast the same 8 variables.

**Resources:**

[anemoi-inference](https://anemoi-inference.readthedocs.io/en/latest/apis/level1.html) package.

In [ ]:
from pathlib import Path
import copy
import subprocess
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch

In [ ]:
from anemoi.inference.checkpoint import Checkpoint
from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from helpers import (
    area_weights,
    find_nearest_point_index,
    patch_ens_single_device_inference,
    patch_graph_attention_backend_for_device,
)

patch_ens_single_device_inference()  # works around a bug in anemoi-models 0.12.0, see helpers.py
patch_graph_attention_backend_for_device()  # lets models trained on GPUs run on a CPU, see helpers.py

In [ ]:
# The only settings that differ between the CPU and the GPU version of this notebook
DEVICE = "cpu"
CKPT_URL = "https://object-store.os-api.cci1.ecmwf.int/ml-tests/test-data/samples/training-course/inference-aifs-crps-o48-6h.ckpt"

## Investigate the checkpoint

For that we first download the checkpoint:

In [ ]:
ckpt_file = Path("../../checkpoints") / Path(CKPT_URL).name
ckpt_file.parent.mkdir(parents=True, exist_ok=True)

# Download the checkpoint (to a temporary file first, so an interrupted download is not mistaken for a checkpoint)
if not ckpt_file.exists():
    print(f"Downloading {CKPT_URL} ...")
    partial_file = ckpt_file.with_suffix(".part")
    subprocess.run(["wget", "-q", CKPT_URL, "-O", str(partial_file)], check=True)
    partial_file.rename(ckpt_file)
print(f"Checkpoint: {ckpt_file} ({ckpt_file.stat().st_size / 1e6:.0f} MB)")

The checkpoint contains all the relevant information about the model and the required input.

In [ ]:
ckpt = Checkpoint(str(ckpt_file))

print(f"Grid: {ckpt.grid} ({ckpt.number_of_grid_points} grid points)")
print(f"Time step: {ckpt.timestep}, input times relative to the start date: {[str(t) for t in ckpt.lagged]}")
print(f"Number of variables: {len(ckpt.variable_categories())}")
print(f"Prognostic variables (input and output): {ckpt.select_variables(include=['prognostic'])}")
print(f"Forcing variables (input only): {ckpt.select_variables(include=['forcing'])}")
print(f"Diagnostic variables (output only): {ckpt.select_variables(include=['diagnostic'])}")

The model architecture

In [ ]:
model = torch.load(ckpt_file, weights_only=False, map_location="cpu")
model

In [ ]:
print(f"Number of weights: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
del model  # the runner below loads its own copy

## Get initial conditions

Running the model for inference requires initial conditions. We retrieve them from [ECMWF Open Data](https://www.ecmwf.int/en/forecasts/datasets/open-data): the IFS high-resolution forecast at step 0 (the analysis) on a 0.25° grid, interpolated to the grid of the model. The model needs two input times, the start date and 6 hours before.

In [ ]:
import datetime
from collections import defaultdict

import numpy as np
import earthkit.data as ekd
import earthkit.regrid as ekr

from ecmwf.opendata import Client as OpendataClient
from helpers import load_saved_state, save_state

In [ ]:
# Initial conditions to download from ECMWF Open Data (must cover the variables the checkpoint needs)
PARAM_SFC = ["2t", "tcw", "lsm", "z", "sdor", "slor"]
PARAM_PL = ["gh", "t"]
LEVELS = [1000, 850, 700, 500, 300, 250]

GRID_RESOLUTION = ckpt.grid
DATE = OpendataClient().latest()
print("Initial date is", DATE)

In [ ]:
def get_fields(dates, param_sfc, param_pl, levels):
    """Download fields from ECMWF Open Data and interpolate them to the model grid.

    Returns a dictionary {name: array of shape (len(dates), grid points)}.
    """
    fields = defaultdict(list)
    for date in dates:
        # levtype must be explicit: some parameters (e.g. z) exist both at the surface and on pressure levels
        for param, levelist, levtype in [(param_sfc, [], "sfc"), (param_pl, levels, "pl")]:
            data = ekd.from_source("ecmwf-open-data", date=date, param=param, levelist=levelist, levtype=levtype)
            for f in data:
                # Open data is between -180 and 180, we need to shift it to 0-360
                assert f.to_numpy().shape == (721, 1440)
                values = np.roll(f.to_numpy(), -f.shape[1] // 2, axis=1)
                # Interpolate the data from 0.25 degrees to the model grid
                values = ekr.interpolate(values, {"grid": (0.25, 0.25)}, {"grid": GRID_RESOLUTION})
                name = f"{f.metadata('param')}_{f.metadata('levelist')}" if levtype == "pl" else f.metadata("param")
                fields[name].append(values)

    # Create a single matrix for each parameter
    fields = {name: np.stack(values) for name, values in fields.items()}

    # Convert geopotential height into geopotential
    for name in [name for name in fields if name.startswith("gh_")]:
        fields[name.replace("gh_", "z_")] = fields.pop(name) * 9.80665

    return fields

Here we make a single forecast from a single initial condition. In operations, the AIFS-ENS is run 51 times, from the control and 50 perturbed initial conditions of the IFS ensemble.

The checkpoint tells us which fields the model needs as input. We check that we have all of them and drop the fields we downloaded but do not need. The downloaded state is cached in a `.npz` file, so asking again for the same date does not download it again.

In [ ]:
def get_input_state(date):
    file = Path(f"inputstate-{GRID_RESOLUTION}-{date.strftime('%Y%m%d_%H')}.npz")
    if file.exists():
        input_state = load_saved_state(file)
        print(f"Input state loaded from file {file}")
    else:
        # The model needs the start date and the date 6 hours before
        fields = get_fields([date - datetime.timedelta(hours=6), date], PARAM_SFC, PARAM_PL, LEVELS)
        input_state = dict(date=date, fields=fields)
        save_state(input_state, file)
        print(f"State for {date} saved to file {file}")

    required = ckpt.select_variables(include=["prognostic", "forcing"])
    missing = set(required) - set(input_state["fields"])
    assert not missing, f"Missing input fields: {sorted(missing)}"
    input_state["fields"] = {name: input_state["fields"][name] for name in required}
    return input_state


input_state = get_input_state(DATE)
print(f"{len(input_state['fields'])} input fields, each with shape {input_state['fields']['2t'].shape} (time, grid points)")

## Run inference from initial conditions

In [ ]:
runner = SimpleRunner(str(ckpt_file), device=DEVICE)

In [ ]:
LEAD_TIME = 48  # hours

### Run the forecast



In [ ]:
ens_members = 1  # This will become important tomorrow
forecast_states = [[] for _ in range(ens_members)]
for member in range(ens_members):
    for state in runner.run(input_state=input_state, lead_time=LEAD_TIME):
        print(f"Member {member}: {state['date']} (lead time: {state['step']})")
        # Store the state if you want to keep it
        forecast_states[member].append(copy.deepcopy(state))

print(f"Forecast completed! Generated {len(forecast_states[0])} time steps.")
print_state(forecast_states[0][-1])

You will see the warning `Coupled forcings are not supported by this runner: ['lsm', 'sdor', 'slor', 'z']`. It is harmless here: these forcings are constant in time, and the `SimpleRunner` keeps the values from the input state instead of reading them again from a data source at every step.

In [ ]:
print(f"Predicted dates: {list(forecast_states[0][x]['date'].strftime('%Y-%m-%dT%H') for x in range(len(forecast_states[0])))}")

Some variables can never be negative: total column water (`tcw`), total precipitation (`tp`) and convective precipitation (`cp`). Does the model respect that?

In [ ]:
for name in ["tcw", "tp", "cp"]:
    minimum = min(state["fields"][name].min() for state in forecast_states[0])
    print(f"{name}: smallest value over all lead times = {minimum:.4g}")

A variable is only guaranteed to be non-negative if the model applies a bounding to it on output, for example a `ReluBounding` (look for `boundings` in the model architecture printed above, and see `model.bounding` in the training configuration). Which variables are bounded in this model? How would you fix the others, at training time or after inference?

# 4. Inspect the generated forecast

### Plot a field

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.tri as tri

In [ ]:
def fix(lons):
    # Shift the longitudes from 0-360 to -180-180
    return np.where(lons > 180, lons - 360, lons)

# pick the last state of the first ensemble member
variable = "t_850"
state = forecast_states[0][-1]
date = state['date']
latitudes = state["latitudes"]
longitudes = fix(state["longitudes"])
values = state["fields"][variable]

# The grid is the same for all fields, so we only need to triangulate it once
triangulation = tri.Triangulation(longitudes, latitudes)
EUROPE = [-25, 45, 34, 72]  # lon min, lon max, lat min, lat max


def plot_field(values, title, label, extent=None, symmetric=False, cmap="RdBu_r"):
    """Plot a field on a map. Use `extent` to zoom in and `symmetric=True` for differences."""
    fig, ax = plt.subplots(figsize=(11, 6), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    if extent is not None:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

    levels = 20
    if symmetric:  # colour scale centred on zero
        vmax = np.abs(values).max()
        levels = np.linspace(-vmax, vmax, 21)

    contour = ax.tricontourf(triangulation, values, levels=levels, transform=ccrs.PlateCarree(), cmap=cmap)
    fig.colorbar(contour, ax=ax, orientation="vertical", shrink=0.7, label=label)
    ax.set_title(title)
    plt.show()

In [ ]:
plot_field(values, f"{variable} at {date}", label=variable)

In [ ]:
plot_field(values, f"{variable} at {date}", label=variable, extent=EUROPE)

### Task 4.1: Plot the 2m temperature forecasts for the weekend?

The forecast is already calculated. Just make sure to select the correct state from the list and select the wanted parameter. The legend must be edited manually. Feel free to convert from Kelvin to Celcius.

### Plot a forecast time-series 

In [ ]:
import pandas as pd

lat_target = 51.45
lon_target = -0.98
idx_nearest = find_nearest_point_index(latitudes, longitudes, lat_target, lon_target)

lead_times = range(len(forecast_states[0]))
n_members = len(forecast_states)

# Extract field for all members and times
values = np.array([
    [forecast_states[m][t]["fields"][variable][idx_nearest] for t in lead_times]
    for m in range(n_members)
])

# Time axis
times = pd.to_datetime([forecast_states[0][t]["date"] for t in lead_times])

# Plot
plt.figure(figsize=(10, 5))

for m in range(n_members):
    plt.plot(times, values[m, :], linewidth=2, label=f"Member {m+1}")

plt.title(f"{variable} forecast at lat={latitudes[idx_nearest]:.2f}, lon={longitudes[idx_nearest]:.2f}")
plt.xlabel("Time")
plt.ylabel(variable)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Task 4.2: Plot a 2m temperature forecast for the location of your city?

We have provided an example for the Reading, UK. Change to the location of the city you live in and select the wanted parameter. 

Extra task: Try to calculate the forecast for more than 48 hours to have an idea of how warm it is going to be next week.

# 5. Sensitivity to the initial conditions

### Task 5: What happens if the initial conditions are wrong?

The model only knows the atmosphere through its initial conditions. Here we put an error in them: a **+5 K warm bias in 2 m temperature over Europe** (35–70°N, 10°W–30°E), at both input times, and look at how the forecast responds.

1. Run the cell below. It creates `biased_state`, a copy of `input_state` with the bias added.
2. Run the forecast again from `biased_state`, with the same `runner` and `LEAD_TIME`.
3. Plot the difference *biased − original* of `2t` at a few lead times (e.g. 6 h, 24 h, 48 h) with `plot_field(..., symmetric=True)`.
4. Plot the mean difference inside the box (`europe` mask) against lead time, for `2t` and `t_850`.

**What to look for:**

- **Memory:** does the +5 K stay, decay or grow? How much of it is left after 48 h? Remember that only `2t` was changed: it is now inconsistent with `t_850` and the geopotential fields.
- **Where it goes:** does the anomaly stay inside the box, get carried downstream by the wind, or show up far away from Europe? The sharp edges of the box are unrealistic: do you see artefacts along them?
- **Other variables:** does `t_850` warm up as well? Physically, a warmer surface layer should slowly warm the air above it.
- **Signal or noise?** Two runs from the same initial conditions already differ, because of the noise the model injects. Is the response to the bias larger than the difference between two unperturbed runs? Tip: call `torch.manual_seed(0)` right before each `runner.run(...)` to give both runs the same noise, so that their difference comes only from the bias.
- **Model comparison:** if you have time, run the same experiment in the other notebook. Do the O48 and the O96 model forget the bias at the same rate?

In [ ]:
# Europe box: 35-70N, 10W-30E (longitudes in -180..180)
europe = (latitudes >= 35) & (latitudes <= 70) & (longitudes >= -10) & (longitudes <= 30)

biased_state = copy.deepcopy(input_state)
biased_state["fields"]["2t"][:, europe] += 5.0  # both input times
print(f"Added +5 K to 2t at {europe.sum()} of {europe.size} grid points")

# 6. Verify a past forecast

How good is the forecast? We cannot verify today's forecast yet, so we start one from `LEAD_TIME` hours ago and compare it with the Open Data analyses at the times it predicts. As a baseline we use *persistence*: the forecast that nothing changes from the initial state.

Open Data only keeps the last few days, so keep `LEAD_TIME` at 48 hours or less for this section.

In [ ]:
past_date = DATE - datetime.timedelta(hours=LEAD_TIME)
past_input_state = get_input_state(past_date)

past_forecast = [copy.deepcopy(state) for state in runner.run(input_state=past_input_state, lead_time=LEAD_TIME)]
valid_dates = [state["date"] for state in past_forecast]
print(f"Forecast from {past_date}, valid from {valid_dates[0]} to {valid_dates[-1]}")

In [ ]:
# Analyses at the times the forecast predicts (one download per valid date)
VERIFY_VARIABLES = {"2t": "K", "t_850": "K", "z_500": "m²/s²"}  # variable: unit
analyses = get_fields(valid_dates, param_sfc=["2t"], param_pl=["gh", "t"], levels=[500, 850])

In [ ]:
# Grid points do not all cover the same area, so the RMSE is area-weighted
weights = area_weights(latitudes)


def rmse(forecast, truth):
    return np.sqrt(np.sum(weights * (forecast - truth) ** 2))


lead_hours = [(d - past_date).total_seconds() / 3600 for d in valid_dates]

fig, axes = plt.subplots(1, len(VERIFY_VARIABLES), figsize=(15, 4))
for ax, (name, unit) in zip(axes, VERIFY_VARIABLES.items()):
    model_rmse = [rmse(state["fields"][name], analyses[name][i]) for i, state in enumerate(past_forecast)]
    persistence_rmse = [rmse(past_input_state["fields"][name][-1], analyses[name][i]) for i in range(len(valid_dates))]
    ax.plot(lead_hours, model_rmse, "o-", label="AIFS")
    ax.plot(lead_hours, persistence_rmse, "s--", label="Persistence")
    ax.set(title=f"RMSE of {name}", xlabel="Lead time (hours)", ylabel=unit)
    ax.grid(True)
axes[0].legend()
plt.tight_layout()
plt.show()

**Things to discuss:** Is the model better than persistence at every lead time? Which variable has the fastest error growth? Why does the `2t` persistence error go up and down with a 24-hour period? The analyses come from the IFS, interpolated to the model grid: what does that mean for the smallest error we can measure?